Librerias

In [49]:
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal
import causalidad as cs

# Escenario 1: Experimento aleatorio

In [50]:
SEED = 1234
TAU  = 1.5

## Simulación de datos

In [51]:
np.random.seed(SEED)

n   = 2000
p   = 0.75
 
# Covariables correlacionadas (slide 18)
mu    = [2, -3]
sigma = [[1.23, -0.2], [-0.2, 0.93]]
X     = multivariate_normal.rvs(mean=mu, cov=sigma, size=n, random_state=SEED)
 
# Tratamiento aleatorio — independiente de X
A = np.random.binomial(1, p, n) 
 
# Outcome
eps = np.random.normal(0, 0.2, n)       # var = 0.04 como en el slide
Y   = 0.25 + TAU * A + 0.3 * X[:,0] + 0.15 * X[:,1] + eps 
 
df = pd.DataFrame({'A': A, 'X1': X[:,0], 'X2': X[:,1], 'Y': Y})
 
tratamiento = 'A'
resultado   = 'Y'
covariables = ['X1', 'X2']
 
df.head()

,A,X1,X2,Y
0,1,1.028472,-3.727337,1.581475
1,1,0.394773,-2.515847,1.632627
2,1,3.104748,-2.648730,2.063955
3,0,0.853992,-3.075344,0.357866
4,0,1.070070,-4.819383,-0.039438


In [52]:
print(f"n = {n}")
print(f"P(A=1) = {A.mean():.3f}  (esperado: {p})")
print(f"\nMedia Y tratados:  {Y[A==1].mean():.4f}")
print(f"Media Y controles: {Y[A==0].mean():.4f}")
print(f"Diferencia naive:  {Y[A==1].mean() - Y[A==0].mean():.4f}")

n = 2000
P(A=1) = 0.741  (esperado: 0.75)

Media Y tratados:  1.9104
Media Y controles: 0.4265
Diferencia naive:  1.4840


In [53]:
print("Balance de covariables (antes de balancear):")
print(f"  Media X1 tratados:  {df.loc[df.A==1,'X1'].mean():.4f}")
print(f"  Media X1 controles: {df.loc[df.A==0,'X1'].mean():.4f}")
print(f"  Media X2 tratados:  {df.loc[df.A==1,'X2'].mean():.4f}")
print(f"  Media X2 controles: {df.loc[df.A==0,'X2'].mean():.4f}")
print()
print("Debe estar balanceada por construcción — las diferencias deben ser pequeñas aunque P(A=1) = 0.75 (grupos de distinto tamaño).")
 

Balance de covariables (antes de balancear):
  Media X1 tratados:  2.0048
  Media X1 controles: 2.0372
  Media X2 tratados:  -2.9637
  Media X2 controles: -2.9753

Debe estar balanceada por construcción — las diferencias deben ser pequeñas aunque P(A=1) = 0.75 (grupos de distinto tamaño).


In [54]:
df = cs.propensity_score(df, tratamiento, covariables)
df[['A', 'X1', 'X2', 'Y', 'propensity_score']].head(10)

propensity_score() — n=2000 (T=1481, C=519)
  PS: min=0.7157  p25=0.7367  media=0.7405  p75=0.7444  max=0.7566


,A,X1,X2,Y,propensity_score
0,1,1.028472,-3.727337,1.581475,0.744304
1,1,0.394773,-2.515847,1.632627,0.749340
2,1,3.104748,-2.648730,2.063955,0.735495
3,0,0.853992,-3.075344,0.357866,0.746193
4,0,1.070070,-4.819383,-0.039438,0.742383
5,1,1.217883,-1.598568,1.728107,0.746678
6,1,0.193121,-4.155365,1.066827,0.747802
7,0,2.345465,-3.170575,0.569643,0.738537
8,0,1.699558,-2.555317,0.377624,0.742773
9,0,0.006963,-3.579125,-0.308976,0.749615


In [55]:
print(f"PS estimado — media: {df['propensity_score'].mean():.4f}  "
      f"std: {df['propensity_score'].std():.4f}")
print(f"PS verdadero = {p} para todas las unidades")
print()

PS estimado — media: 0.7405  std: 0.0059
PS verdadero = 0.75 para todas las unidades



## ATE - Sin balancear

In [56]:
res_orig = cs.calcular_ate(df, resultado, tratamiento, covariables)
res_orig
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_orig[key]
    err = val - TAU
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}")
 
print()
print("Todos deberían dar ≈ 1.5 porque A ⊥⊥ X.")

τ verdadero = 1.5
Estimador           ATE     Error
────────────────────────────────────
naive             1.484    -0.016
regresion         1.492    -0.008
g_formula         1.492    -0.008
ht                1.492    -0.008
hajek             1.492    -0.008
msm               1.492    -0.008
dr                1.492    -0.008

Todos deberían dar ≈ 1.5 porque A ⊥⊥ X.


## ATE - Balanceado

### Trimming


In [57]:
df_trim = cs.balance(df, tratamiento, metodo='trimming', kappa=0.05)
 
print(f"Filas antes: {len(df)}")
print(f"Filas después del trimming: {len(df_trim)}")
print(f"Eliminadas: {len(df) - len(df_trim)}")
print()
print(f"PS en df_trim — min: {df_trim['propensity_score'].min():.4f}  "
      f"max: {df_trim['propensity_score'].max():.4f}")
print()
print("Con un RCT el trimming no debería eliminar a nadie porque el PS")
print("no tiene valores extremos.")
 
res_trim = cs.calcular_ate(df_trim, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_trim[key]
    err = val - TAU
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}")


──────────────────────────────────────────────────
  balance() | metodo='trimming' | n=2000 (T=1481, C=519)
──────────────────────────────────────────────────
Trimming (kappa=0.05) — 0 unidades eliminadas (0.0%)  -> n final=2000  (tratados=1481, controles=519)
  Rango PS conservado: [0.7157, 0.7566]
──────────────────────────────────────────────────

Filas antes: 2000
Filas después del trimming: 2000
Eliminadas: 0

PS en df_trim — min: 0.7157  max: 0.7566

Con un RCT el trimming no debería eliminar a nadie porque el PS
no tiene valores extremos.
τ verdadero = 1.5
Estimador           ATE     Error
────────────────────────────────────
naive             1.484    -0.016
regresion         1.492    -0.008
g_formula         1.492    -0.008
ht                1.492    -0.008
hajek             1.492    -0.008
msm               1.492    -0.008
dr                1.492    -0.008


### Truncating

In [58]:
df_trunc = cs.balance(df, tratamiento, metodo='truncating', kappa=0.05)
 
print(f"Filas: {len(df_trunc)}  (sin eliminar)")
print(f"PS original  — min: {df_trunc['propensity_score_original'].min():.4f}  "
      f"max: {df_trunc['propensity_score_original'].max():.4f}")
print(f"PS truncado  — min: {df_trunc['propensity_score'].min():.4f}  "
      f"max: {df_trunc['propensity_score'].max():.4f}")
 
res_trunc = cs.calcular_ate(df_trunc, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_trunc[key]
    err = val - TAU
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}")


──────────────────────────────────────────────────
  balance() | metodo='truncating' | n=2000 (T=1481, C=519)
──────────────────────────────────────────────────
Truncating (kappa=0.05) — 0 PS truncados (0.0%)  -> n=2000 (sin eliminar filas)
  PS original:  [0.7157, 0.7566]  ->  PS truncado: [0.7157, 0.7566]
──────────────────────────────────────────────────

Filas: 2000  (sin eliminar)
PS original  — min: 0.7157  max: 0.7566
PS truncado  — min: 0.7157  max: 0.7566
τ verdadero = 1.5
Estimador           ATE     Error
────────────────────────────────────
naive             1.484    -0.016
regresion         1.492    -0.008
g_formula         1.492    -0.008
ht                1.492    -0.008
hajek             1.492    -0.008
msm               1.492    -0.008
dr                1.492    -0.008


### Matching

In [59]:
df_match = cs.balance(df, tratamiento, metodo='matching')
 
print(f"Filas antes:  {len(df)}")
print(f"Filas después del matching: {len(df_match)}")
print()
print(f"Pares formados: {len(df_match)//2}")
print(f"n tratados: {(df_match[tratamiento]==1).sum()}")
print(f"n controles: {(df_match[tratamiento]==0).sum()}")
 
res_match = cs.calcular_ate(df_match, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_match[key]
    err = val - TAU
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}")


──────────────────────────────────────────────────
  balance() | metodo='matching' | n=2000 (T=1481, C=519)
──────────────────────────────────────────────────
Matching 1:1 — 519 pares  (tratados=519, controles=519)
──────────────────────────────────────────────────

Filas antes:  2000
Filas después del matching: 1038

Pares formados: 519
n tratados: 519
n controles: 519
τ verdadero = 1.5
Estimador           ATE     Error
────────────────────────────────────
naive             1.463    -0.037
regresion         1.496    -0.004
g_formula         1.496    -0.004
ht                0.466    -1.034
hajek             1.471    -0.029
msm               1.471    -0.029
dr                1.496    -0.004


### Subclasification

In [60]:
df_sub = cs.balance(df, tratamiento, metodo='subclassif', n_subclases=5)
 
print("Distribución por subclase:")
for sc in sorted(df_sub['subclase'].dropna().unique()):
    sub = df_sub[df_sub['subclase'] == sc]
    nt  = (sub[tratamiento] == 1).sum()
    nc  = (sub[tratamiento] == 0).sum()
    print(f"  Subclase {int(sc)}: n={len(sub):4d}  T={nt:4d}  C={nc:4d}  "
          f"PS_medio={sub['propensity_score'].mean():.3f}")
 
res_sub = cs.calcular_ate(df_sub, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_sub[key]
    err = val - TAU
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}")


──────────────────────────────────────────────────
  balance() | metodo='subclassif' | n=2000 (T=1481, C=519)
──────────────────────────────────────────────────
Subclasificación — 5 subclases (solicitadas: 5)
  Subclase  1: n= 400  tratados=293  controles=107  PS_medio=0.732
  Subclase  2: n= 400  tratados=293  controles=107  PS_medio=0.737
  Subclase  3: n= 400  tratados=297  controles=103  PS_medio=0.741
  Subclase  4: n= 400  tratados=300  controles=100  PS_medio=0.744
  Subclase  5: n= 400  tratados=298  controles=102  PS_medio=0.749
──────────────────────────────────────────────────

Distribución por subclase:
  Subclase 1: n= 400  T= 293  C= 107  PS_medio=0.732
  Subclase 2: n= 400  T= 293  C= 107  PS_medio=0.737
  Subclase 3: n= 400  T= 297  C= 103  PS_medio=0.741
  Subclase 4: n= 400  T= 300  C= 100  PS_medio=0.744
  Subclase 5: n= 400  T= 298  C= 102  PS_medio=0.749
τ verdadero = 1.5
Estimador           ATE     Error
────────────────────────────────────
naive             1.48

# Escenario 2: Tratamiento es afectado por covariables

In [61]:
SEED = 1234
TAU  = 1.0

## Simulación de datos

In [62]:
n = 2000
 
# Covariables correlacionadas (slide 37)
mu    = [1, 2]
sigma = [[0.750, -0.375], [-0.375, 1.0]]
X     = multivariate_normal.rvs(mean=mu, cov=sigma, size=n, random_state=SEED)
 
# Mecanismo de asignación — depende de X
eta  = 0.8 * X[:,0] - 0.5 * X[:,1]
ps_v = 1 / (1 + np.exp(-eta))          # PS verdadero
A    = np.random.binomial(1, ps_v)
 
# Outcome — X1 y X2 son confusores
eps = np.random.normal(0, 0.75, n)
Y   = 0.75 + TAU * A + 1.4 * X[:,0] + 5.0 * X[:,1] + eps
 
df = pd.DataFrame({
    'A'       : A,
    'X1'      : X[:,0],
    'X2'      : X[:,1],
    'Y'       : Y,
    'ps_verd' : ps_v       # guardamos el PS verdadero para comparar
})
 
tratamiento = 'A'
resultado   = 'Y'
covariables = ['X1', 'X2']
 
df.head()

,A,X1,X2,Y,ps_verd
0,0,0.020137,1.948725,10.311912,0.277228
1,0,-0.119839,3.183344,17.184227,0.156099
2,1,1.973352,1.700429,14.783930,0.674469
3,0,0.075875,2.528165,13.942228,0.230874
4,0,-0.270460,1.106111,8.086520,0.316604


In [ ]:
print(f"n = {n}")
print(f"P(A=1) = {A.mean():.3f}  (ya no es fijo — depende de X)")
print()
print("Desbalance de covariables:")
print(f"  Media X1 tratados:  {df.loc[df.A==1,'X1'].mean():.4f}")
print(f"  Media X1 controles: {df.loc[df.A==0,'X1'].mean():.4f}  "
      f"← diferencia: {df.loc[df.A==1,'X1'].mean()-df.loc[df.A==0,'X1'].mean():+.4f}")
print(f"  Media X2 tratados:  {df.loc[df.A==1,'X2'].mean():.4f}")
print(f"  Media X2 controles: {df.loc[df.A==0,'X2'].mean():.4f}  "
      f"← diferencia: {df.loc[df.A==1,'X2'].mean()-df.loc[df.A==0,'X2'].mean():+.4f}")
print()
print("A diferencia de antes, aquí los grupos son distintos en X.")
print("Esa diferencia se va a traducir en sesgo si no la controlamos.")
 
# %%
# El PS verdadero ya varía — a diferencia del RCT donde era 0.75 para todos
print("PS verdadero (conocido porque simulamos nosotros):")
print(f"  min={ps_v.min():.4f}  p25={np.percentile(ps_v,25):.4f}  "
      f"media={ps_v.mean():.4f}  p75={np.percentile(ps_v,75):.4f}  "
      f"max={ps_v.max():.4f}")
print()
print("PS verdadero por grupo:")
print(f"  Tratados  (A=1): media={ps_v[A==1].mean():.4f}  "
      f"← tienen PS más alto, era más probable que recibieran tratamiento")
print(f"  Controles (A=0): media={ps_v[A==0].mean():.4f}  "
      f"← tienen PS más bajo")
 
# %%
# Diferencia naive — sesgada
naive_bruto = Y[A==1].mean() - Y[A==0].mean()
print(f"Diferencia naive = {naive_bruto:.4f}")
print(f"τ verdadero      = {TAU:.4f}")
print(f"Sesgo naive      = {naive_bruto - TAU:+.4f}")
print()
print("El naive estima mal porque los tratados también tienen X1 y X2 más")
print("altos, y X1 y X2 suben Y directamente (coefs 1.4 y 5.0).")

n = 2000
P(A=1) = 0.451  (ya no es fijo — depende de X)

Desbalance de covariables:
  Media X1 tratados:  1.3737
  Media X1 controles: 0.7305  ← diferencia: +0.6431
  Media X2 tratados:  1.6847
  Media X2 controles: 2.2918  ← diferencia: -0.6071

A diferencia de antes, aquí los grupos son distintos en X.
Esa diferencia se va a traducir en sesgo si no la controlamos.
PS verdadero (conocido porque simulamos nosotros):
  min=0.0474  p25=0.2998  media=0.4597  p75=0.6095  max=0.9794

PS verdadero por grupo:
  Tratados  (A=1): media=0.5533  ← tienen PS más alto, era más probable que recibieran tratamiento
  Controles (A=0): media=0.3830  ← tienen PS más bajo
Diferencia naive = -1.1250
τ verdadero      = 1.0000
Sesgo naive      = -2.1250

El naive sobreestima porque los tratados también tienen X1 y X2 más
altos, y X1 y X2 suben Y directamente (coefs 1.4 y 5.0).


In [64]:
df = cs.propensity_score(df, tratamiento, covariables)
 
# %%
# Comparar PS estimado vs PS verdadero
print("PS estimado:")
print(f"  min={df['propensity_score'].min():.4f}  "
      f"p25={df['propensity_score'].quantile(0.25):.4f}  "
      f"media={df['propensity_score'].mean():.4f}  "
      f"p75={df['propensity_score'].quantile(0.75):.4f}  "
      f"max={df['propensity_score'].max():.4f}")
print()
print("PS verdadero:")
print(f"  min={ps_v.min():.4f}  "
      f"p25={np.percentile(ps_v,25):.4f}  "
      f"media={ps_v.mean():.4f}  "
      f"p75={np.percentile(ps_v,75):.4f}  "
      f"max={ps_v.max():.4f}")
print()
print("El PS estimado debería parecerse mucho al verdadero porque el modelo")
print("logístico está correctamente especificado (incluye X1 y X2).")

propensity_score() — n=2000 (T=901, C=1099)
  PS: min=0.0431  p25=0.2873  media=0.4505  p75=0.5995  max=0.9792
  Aviso: 8 unidades (0.4%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
PS estimado:
  min=0.0431  p25=0.2873  media=0.4505  p75=0.5995  max=0.9792

PS verdadero:
  min=0.0474  p25=0.2998  media=0.4597  p75=0.6095  max=0.9794

El PS estimado debería parecerse mucho al verdadero porque el modelo
logístico está correctamente especificado (incluye X1 y X2).


In [66]:
corr = np.corrcoef(df['propensity_score'], df['ps_verd'])[0,1]
print(f"Correlación PS estimado vs PS verdadero: {corr:.4f}")
print("Una correlación alta (>0.99) indica que el modelo logístico")
print("capturó bien el mecanismo de asignación.")

Correlación PS estimado vs PS verdadero: 0.9995
Una correlación alta (>0.99) indica que el modelo logístico
capturó bien el mecanismo de asignación.



## ATE - Sin balancear

In [67]:
res_orig = cs.calcular_ate(df, resultado, tratamiento, covariables)
res_orig
 
# %%
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_orig[key]
    err = val - TAU
    flag = '  ← SESGO' if abs(err) > 0.15 else ''
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}{flag}")
 
print()
print("El naive debería estar sesgado.")
print("Regresión, g-fórmula y DR deberían estar cerca de 1.0 porque")
print("controlan X1 y X2 explícitamente.")
print("HT y Hajek dependen del PS estimado — si el PS es bueno, convergen.")

τ verdadero = 1.0
Estimador           ATE     Error
────────────────────────────────────
naive            -1.125    -2.125  ← SESGO
regresion         1.022    +0.022
g_formula         1.022    +0.022
ht                0.719    -0.281  ← SESGO
hajek             0.942    -0.058
msm               0.942    -0.058
dr                1.037    +0.037

El naive debería estar sesgado.
Regresión, g-fórmula y DR deberían estar cerca de 1.0 porque
controlan X1 y X2 explícitamente.
HT y Hajek dependen del PS estimado — si el PS es bueno, convergen.


## ATE - Balanceado

### Trimming

In [68]:
df_trim = cs.balance(df, tratamiento, metodo='trimming', kappa=0.05)
 
# %%
print(f"Filas antes:  {len(df)}")
print(f"Filas después: {len(df_trim)}")
print(f"Eliminadas:   {len(df) - len(df_trim)}")
print()
print("A diferencia del RCT, aquí sí hay PS extremos porque A depende de X.")
print("El trimming elimina las unidades sin contrafactual comparable.")
 
# %%
# Ver si el desbalance en X mejoró después del trimming
print("Desbalance de covariables DESPUÉS del trimming:")
print(f"  Media X1 tratados:  {df_trim.loc[df_trim.A==1,'X1'].mean():.4f}")
print(f"  Media X1 controles: {df_trim.loc[df_trim.A==0,'X1'].mean():.4f}  "
      f"← diferencia: {df_trim.loc[df_trim.A==1,'X1'].mean()-df_trim.loc[df_trim.A==0,'X1'].mean():+.4f}")
print(f"  Media X2 tratados:  {df_trim.loc[df_trim.A==1,'X2'].mean():.4f}")
print(f"  Media X2 controles: {df_trim.loc[df_trim.A==0,'X2'].mean():.4f}  "
      f"← diferencia: {df_trim.loc[df_trim.A==1,'X2'].mean()-df_trim.loc[df_trim.A==0,'X2'].mean():+.4f}")
 
# %%
res_trim = cs.calcular_ate(df_trim, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_trim[key]
    err = val - TAU
    flag = '  ← SESGO' if abs(err) > 0.15 else ''
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}{flag}")


──────────────────────────────────────────────────
  balance() | metodo='trimming' | n=2000 (T=901, C=1099)
──────────────────────────────────────────────────
Trimming (kappa=0.05) — 8 unidades eliminadas (0.4%)  -> n final=1992  (tratados=898, controles=1094)
  Rango PS conservado: [0.0502, 0.9494]
──────────────────────────────────────────────────

Filas antes:  2000
Filas después: 1992
Eliminadas:   8

A diferencia del RCT, aquí sí hay PS extremos porque A depende de X.
El trimming elimina las unidades sin contrafactual comparable.
Desbalance de covariables DESPUÉS del trimming:
  Media X1 tratados:  1.3665
  Media X1 controles: 0.7406  ← diferencia: +0.6259
  Media X2 tratados:  1.6939
  Media X2 controles: 2.2861  ← diferencia: -0.5923
τ verdadero = 1.0
Estimador           ATE     Error
────────────────────────────────────
naive            -1.076    -2.076  ← SESGO
regresion         1.023    +0.023
g_formula         1.023    +0.023
ht                0.762    -0.238  ← SESGO
hajek

### Truncating

In [69]:
df_trunc = cs.balance(df, tratamiento, metodo='truncating', kappa=0.05)
 
# %%
print(f"Filas: {len(df_trunc)}  (sin eliminar)")
print()
n_truncados = (df['propensity_score'] < 0.05).sum() + \
              (df['propensity_score'] > 0.95).sum()
print(f"PS truncados: {n_truncados}")
print(f"PS original  — min: {df_trunc['propensity_score_original'].min():.4f}  "
      f"max: {df_trunc['propensity_score_original'].max():.4f}")
print(f"PS truncado  — min: {df_trunc['propensity_score'].min():.4f}  "
      f"max: {df_trunc['propensity_score'].max():.4f}")
 
# %%
res_trunc = cs.calcular_ate(df_trunc, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_trunc[key]
    err = val - TAU
    flag = '  ← SESGO' if abs(err) > 0.15 else ''
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}{flag}")


──────────────────────────────────────────────────
  balance() | metodo='truncating' | n=2000 (T=901, C=1099)
──────────────────────────────────────────────────
Truncating (kappa=0.05) — 8 PS truncados (0.4%)  -> n=2000 (sin eliminar filas)
  PS original:  [0.0431, 0.9792]  ->  PS truncado: [0.0500, 0.9500]
──────────────────────────────────────────────────

Filas: 2000  (sin eliminar)

PS truncados: 8
PS original  — min: 0.0431  max: 0.9792
PS truncado  — min: 0.0500  max: 0.9500
τ verdadero = 1.0
Estimador           ATE     Error
────────────────────────────────────
naive            -1.125    -2.125  ← SESGO
regresion         1.022    +0.022
g_formula         1.022    +0.022
ht                0.719    -0.281  ← SESGO
hajek             0.942    -0.058
msm               0.942    -0.058
dr                1.037    +0.037


### Matching

In [70]:
df_match = cs.balance(df, tratamiento, metodo='matching')
 
# %%
print(f"Filas antes:  {len(df)}")
print(f"Filas después: {len(df_match)}")
print(f"Pares formados: {len(df_match)//2}")
print()
# Desbalance después del matching
print("Desbalance de covariables DESPUÉS del matching:")
print(f"  Media X1 tratados:  {df_match.loc[df_match.A==1,'X1'].mean():.4f}")
print(f"  Media X1 controles: {df_match.loc[df_match.A==0,'X1'].mean():.4f}  "
      f"← diferencia: {df_match.loc[df_match.A==1,'X1'].mean()-df_match.loc[df_match.A==0,'X1'].mean():+.4f}")
print(f"  Media X2 tratados:  {df_match.loc[df_match.A==1,'X2'].mean():.4f}")
print(f"  Media X2 controles: {df_match.loc[df_match.A==0,'X2'].mean():.4f}  "
      f"← diferencia: {df_match.loc[df_match.A==1,'X2'].mean()-df_match.loc[df_match.A==0,'X2'].mean():+.4f}")
print()
print("El matching debería reducir el desbalance en X acercando los controles")
print("a los tratados en términos de PS.")
 
# %%
res_match = cs.calcular_ate(df_match, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_match[key]
    err = val - TAU
    flag = '  ← SESGO' if abs(err) > 0.15 else ''
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}{flag}")


──────────────────────────────────────────────────
  balance() | metodo='matching' | n=2000 (T=901, C=1099)
──────────────────────────────────────────────────
Matching 1:1 — 901 pares  (tratados=901, controles=901)
──────────────────────────────────────────────────

Filas antes:  2000
Filas después: 1802
Pares formados: 901

Desbalance de covariables DESPUÉS del matching:
  Media X1 tratados:  1.3737
  Media X1 controles: 0.9547  ← diferencia: +0.4190
  Media X2 tratados:  1.6847
  Media X2 controles: 2.0821  ← diferencia: -0.3975

El matching debería reducir el desbalance en X acercando los controles
a los tratados en términos de PS.
τ verdadero = 1.0
Estimador           ATE     Error
────────────────────────────────────
naive            -0.378    -1.378  ← SESGO
regresion         1.025    +0.025
g_formula         1.025    +0.025
ht                2.910    +1.910  ← SESGO
hajek             1.505    +0.505  ← SESGO
msm               1.505    +0.505  ← SESGO
dr                1.046    

### Subclasificación

In [71]:
df_sub = cs.balance(df, tratamiento, metodo='subclassif', n_subclases=5)
 
# %%
print("Distribución por subclase:")
for sc in sorted(df_sub['subclase'].dropna().unique()):
    sub = df_sub[df_sub['subclase'] == sc]
    nt  = (sub[tratamiento] == 1).sum()
    nc  = (sub[tratamiento] == 0).sum()
    print(f"  Subclase {int(sc)}: n={len(sub):4d}  T={nt:4d}  C={nc:4d}  "
          f"PS_medio={sub['propensity_score'].mean():.3f}")
 
print()
print("A diferencia del RCT donde las subclases tenían ~50/50 tratados/controles,")
print("aquí la subclase 1 (PS bajo) tiene pocos tratados y la subclase 5")
print("(PS alto) tiene pocos controles — el desbalance original es visible.")
 
# %%
res_sub = cs.calcular_ate(df_sub, resultado, tratamiento, covariables)
 
print(f"τ verdadero = {TAU}")
print(f"{'Estimador':<14} {'ATE':>8}  {'Error':>8}")
print(f"{'─'*36}")
for key in ['naive', 'regresion', 'g_formula', 'ht', 'hajek', 'msm', 'dr']:
    val = res_sub[key]
    err = val - TAU
    flag = '  ← SESGO' if abs(err) > 0.15 else ''
    print(f"{key:<14} {val:>8.3f}  {err:>+8.3f}{flag}")


──────────────────────────────────────────────────
  balance() | metodo='subclassif' | n=2000 (T=901, C=1099)
──────────────────────────────────────────────────
Subclasificación — 5 subclases (solicitadas: 5)
  Subclase  1: n= 400  tratados= 61  controles=339  PS_medio=0.172
  Subclase  2: n= 400  tratados=135  controles=265  PS_medio=0.319
  Subclase  3: n= 400  tratados=176  controles=224  PS_medio=0.440
  Subclase  4: n= 400  tratados=231  controles=169  PS_medio=0.568
  Subclase  5: n= 400  tratados=298  controles=102  PS_medio=0.753
──────────────────────────────────────────────────

Distribución por subclase:
  Subclase 1: n= 400  T=  61  C= 339  PS_medio=0.172
  Subclase 2: n= 400  T= 135  C= 265  PS_medio=0.319
  Subclase 3: n= 400  T= 176  C= 224  PS_medio=0.440
  Subclase 4: n= 400  T= 231  C= 169  PS_medio=0.568
  Subclase 5: n= 400  T= 298  C= 102  PS_medio=0.753

A diferencia del RCT donde las subclases tenían ~50/50 tratados/controles,
aquí la subclase 1 (PS bajo) tiene 